# LegalQA Main 1/3 — QLoRA train và chạy tiếp trên Kaggle

Chọn **GPU T4 x2**. Add Input dataset BTC và Version 3 chứa `index/` + `models/`.
Notebook tự tìm dưới `/kaggle/input`, không phụ thuộc tên/mức lồng thư mục của Kaggle.

**Chạy tiếp:** Save Version có output; lần sau Add Input **toàn bộ output notebook lần trước**, giữ Input Version 3 và để `INPUT_MODE='auto'`. Tự khóa commit, kiểm tra checksum/fingerprint, copy tiến độ sang `/kaggle/working`, dùng lại retrieval hoàn tất (hoặc journal dở dang), chọn checkpoint hợp lệ có global_step cao nhất và khôi phục optimizer/RNG cả hai rank. Không cần dataset gốc nếu output đã có split đầy đủ.

`auto` ưu tiên resume output Stage 1; chỉ có diagnostics ZIP thì import retrieval cho lượt train mới. `resume` bắt buộc có output; `retrieval` tự tìm cache cũ/ZIP, dùng code mới và bắt đầu optimizer mới; `fresh` bỏ qua output cũ. Bỏ qua diagnostics thiếu trọng số/optimizer. Nhiều output đầy đủ: ưu tiên `PREFERRED_PREVIOUS_NOTEBOOK` đã cấu hình; nếu vẫn không xác định được thì dừng để chọn ROOT. `PREVIOUS_OUTPUT` cụ thể luôn được ưu tiên. Cache 768 QA không được trộn vào lượt 5.600 QA.

**RAM/GPU:** code mới đọc cache từng bản ghi, nén token ngay khi tạo, nạp hai model lần lượt; QLoRA NF4, fused loss, checkpointing, batch 1/GPU × accumulation 4 × 2 GPU = 8. Giữ toàn bộ target và giới hạn 8192 token. Dừng/lưu khi RAM thấp; supervisor dừng cả nhóm worker dưới ngưỡng khẩn cấp. Không thể cam kết không OOM trước khi đo trên dữ liệu/GPU thật. BM25 là tác vụ CPU; DDP proof xác nhận optimizer chạy trên cả hai GPU.

**Commit cũ:** resume giữ code của checkpoint để bảo toàn tiến độ; không tự ghép bản sửa loss/RAM vào optimizer cũ. Muốn áp dụng code mới cho output khác commit, dùng `INPUT_MODE='retrieval'` (train lại, tái sử dụng BM25). Push bản sửa lên repo được clone trước khi bắt đầu lượt mới.

Tối đa 9 giờ từ cell đầu + 10 phút export. `paused` là snapshot hợp lệ; diagnostics ZIP chỉ phục vụ kiểm tra, không có trọng số. Chỉ sang Stage 2 khi `STATUS: complete`. Chi tiết và smoke test: `docs/main01_resume.md`.

**Resume bitsandbytes:** notebook áp dụng bản vá runtime cho bitsandbytes 0.45.5 khi nạp optimizer đã lưu: bỏ metadata paged không còn hợp lệ, giữ nguyên moment/step và đưa state về GPU của rank tương ứng. Chạy smoke save/load optimizer trên cả hai GPU trước model lớn; phải thấy hai dòng `BNB_RESUME_SMOKE_OK`. Bản vá không sửa file code/config/checkpoint được ghim; báo cáo nằm trong `runtime_compat/bnb_resume_rank*.json`. State được phục hồi dùng VRAM thông thường thay cho paging; log ghi chính xác số byte.

Input test: `private-official.json` (1.918 câu). Dùng output private mới; không resume output public cũ. Tên artifact `public` trong workflow Stage 1–4 là tên nội bộ được giữ để tương thích.


In [1]:
import json, os, signal, subprocess, sys, time
from pathlib import Path

# Count setup/install time too. Do not reset this timestamp in later cells.
SESSION_STARTED = time.monotonic()
if not Path('/kaggle').is_dir():
    raise RuntimeError('Notebook chỉ chạy trên Kaggle.')
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
CODE = WORK / 'legalqa_stage_code'

# 9h includes setup and compute; export gets up to 10 additional minutes.
# The remaining margin is reserved for Kaggle output collection and runtime variation.
WORK_HOURS = 9.0
EXPORT_SECONDS = 600
MIN_FREE_RAM_MB = 3072  # Cooperative save threshold; hard stop below 1536 MiB.
VERSION3_ROOT = None     # Auto-find index + models anywhere under /kaggle/input.
DATASET_ROOT = None      # Auto-find train.json + private-official.json; optional on resume.

# Auto-discover complete snapshots; explicit PREVIOUS_OUTPUT always takes precedence.
INPUT_MODE = 'auto'     # auto / resume / retrieval / fresh; auto prioritizes full output.
PREVIOUS_OUTPUT = None   # Cumulative output of THIS stage from an earlier session.
PREFERRED_PREVIOUS_NOTEBOOK = 'lighth/legalqa-main-01-qlora-train'  # User-selected source; None disables preference.
UPSTREAM_OUTPUT = None   # Stage 1 for notebook 02; Stage 2 for notebook 03.
LEGACY_INPUT_ROOT = None # Notebook 01 only: old legalqa_quality_v8_full with completed QLoRA.
RETRIEVAL_INPUT = None   # Stage 1: old output folder or diagnostics ZIP; reuse retrieval only.
REPO_REVISION = None     # First run: main. Continuations: automatically pin upstream commit.

STAGE = 1
MODE = 'auto'
MAX_NEW_QUESTIONS = 200

# Fail before retrieval if the requested Kaggle accelerator is unavailable.
gpu_names = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True, timeout=30
).strip().splitlines()
if len(gpu_names) != 2 or not all('T4' in name for name in gpu_names):
    raise RuntimeError(f'Chọn accelerator GPU T4 x2 trước khi chạy Stage 1; hiện có: {gpu_names}')
print('Training GPUs:', gpu_names)


Training GPUs: ['Tesla T4', 'Tesla T4']


## Tự tìm input, khóa commit và kiểm soát phiên

Giữ một output của lượt cần resume trong Input. `PREVIOUS_OUTPUT` nhận cả ROOT hoặc thư mục mount bao ngoài.


In [2]:
if not 0 < WORK_HOURS <= 9:
    raise ValueError('WORK_HOURS phải trong (0, 9]; giữ thời gian dự phòng trước 12h.')
WORK_END = SESSION_STARTED + WORK_HOURS * 3600

def find_root(value, marker_name, required_files, label, required=True):
    if value is not None:
        explicit = Path(value)
        if not explicit.exists():
            raise FileNotFoundError(explicit)
        search = explicit.parent if explicit.is_file() else explicit
    else:
        search = INPUT
    candidates = sorted({p.parent for p in search.rglob(marker_name)
                         if all((p.parent/name).is_file() for name in required_files)})
    if len(candidates) > 1:
        raise RuntimeError(f'Nhiều {label}: {candidates}. Chỉ định ROOT trong cell cấu hình.')
    if candidates:
        return candidates[0]
    if required or value is not None:
        raise FileNotFoundError(f'Không tìm thấy {label} trong {search}; cần {required_files}')
    return None

def snapshot_problem(root, stage):
    # Structural preflight only; Stage.restore still verifies every SHA256.
    marker = root/f'stage{stage}_manifest.json'
    try:
        info = json.loads(marker.read_text(encoding='utf-8-sig'))
        files = info.get('files', {})
        if info.get('schema') != 2 or info.get('stage') != stage or not isinstance(files, dict):
            return 'manifest không phải snapshot schema 2 của stage này'
        required = {'config.json', 'session.json', 'models.lock.json'}
        if stage != 1 or info.get('status') == 'complete' or 'sft/training_manifest.json' in files:
            required.add('data/split_manifest.json')
        if not required.issubset(files):
            return 'thiếu provenance của output đầy đủ'
        resolved = root.resolve()
        for name, expected in files.items():
            artifact = (root/name).resolve()
            if resolved not in artifact.parents:
                return f'artifact nằm ngoài ROOT: {name}'
            if not artifact.is_file() or artifact.stat().st_size != expected['size']:
                return f'thiếu file hoặc sai kích thước: {name}'
    except (OSError, ValueError, TypeError, KeyError, AttributeError) as error:
        return f'manifest/artifact không đọc được: {error}'
    return None

def resolve_output(value, stage, required=False):
    marker = f'stage{stage}_manifest.json'
    if value is not None:
        explicit = Path(value)
        if not explicit.exists():
            raise FileNotFoundError(explicit)
        search = explicit.parent if explicit.is_file() else explicit
        # A full ROOT must not accidentally match diagnostics nested below it.
        if (search/marker).is_file():
            problem = snapshot_problem(search, stage)
            if problem:
                raise ValueError(f'PREVIOUS_OUTPUT không đủ để resume: {search}: {problem}. Cần output đầy đủ, không phải diagnostics.')
            return search
    else:
        search = INPUT
    candidates, rejected = [], []
    for manifest in sorted(search.rglob(marker)):
        root = manifest.parent
        problem = snapshot_problem(root, stage)
        if problem:
            rejected.append((root, problem))
            print(f'Không dùng để resume: {root}: {problem}', flush=True)
        else:
            candidates.append(root)
    preferred = PREFERRED_PREVIOUS_NOTEBOOK if value is None else None
    if len(candidates) > 1 and preferred:
        owner, slug = preferred.split('/')
        preferred_roots = [root for root in candidates
                           if root.relative_to(INPUT).parts[:3] == ('notebooks', owner, slug)
                           or root.relative_to(INPUT).parts[:1] == (slug,)]
        if len(preferred_roots) == 1:
            print(f'Ưu tiên output notebook đã chọn {preferred}: {preferred_roots[0]}', flush=True)
            return preferred_roots[0]
    if len(candidates) > 1:
        raise RuntimeError(f'Nhiều output Stage {stage} đầy đủ: {candidates}. Chỉ định PREVIOUS_OUTPUT hoặc PREFERRED_PREVIOUS_NOTEBOOK.')
    if candidates:
        return candidates[0]
    if rejected:
        raise ValueError(f'Có manifest Stage {stage} nhưng không có output đủ để resume: {rejected}. Gắn output đầy đủ; chỉ muốn dùng retrieval thì đặt INPUT_MODE="retrieval".')
    if required or value is not None:
        raise FileNotFoundError(f'Không tìm thấy output Stage {stage} trong {search}.')
    return None

def resolve_retrieval(value=None):
    from zipfile import ZipFile, BadZipFile
    search = Path(value) if value is not None else INPUT
    if not search.exists():
        raise FileNotFoundError(search)
    if search.is_file() and search.suffix.lower() != '.zip':
        search = search.parent
    sources = []
    manifests = [search/'stage1_manifest.json'] if search.is_dir() and (search/'stage1_manifest.json').is_file() else list(search.rglob('stage1_manifest.json')) if search.is_dir() else []
    for manifest in manifests:
        info = json.loads(manifest.read_text(encoding='utf-8'))
        name = 'train.sft.lexical.retrieval.json'
        if name in info.get('files', {}) and (manifest.parent/name).is_file():
            sources.append(manifest.parent)
    # Do not count a diagnostics ZIP twice when the full folder is attached.
    archives = [search] if search.is_file() else sorted(search.rglob('*.zip'))
    for archive in archives:
        if any(root == archive.parent or root in archive.parents for root in sources):
            continue
        try:
            with ZipFile(archive) as z:
                names = z.namelist()
                if 'stage1_manifest.json' in names and 'train.sft.lexical.retrieval.json' in names:
                    sources.append(archive)
        except BadZipFile:
            continue
    if len(sources) != 1:
        raise RuntimeError(f'Cần đúng một nguồn retrieval hoàn tất, tìm thấy: {sources}. Chỉ định RETRIEVAL_INPUT.')
    return sources[0]

if INPUT_MODE not in {'auto', 'resume', 'retrieval', 'fresh'}:
    raise ValueError('INPUT_MODE phải là auto/resume/retrieval/fresh.')
if UPSTREAM_OUTPUT is not None:
    raise ValueError('Stage 1 không nhận UPSTREAM_OUTPUT.')
if INPUT_MODE == 'fresh':
    if any(v is not None for v in (PREVIOUS_OUTPUT, RETRIEVAL_INPUT, LEGACY_INPUT_ROOT)):
        raise ValueError('fresh không được trộn với previous/retrieval/legacy.')
elif RETRIEVAL_INPUT is not None or INPUT_MODE == 'retrieval':
    if INPUT_MODE == 'resume' or PREVIOUS_OUTPUT is not None or LEGACY_INPUT_ROOT is not None:
        raise ValueError('Retrieval-only import không resume checkpoint; bỏ previous/legacy.')
    RETRIEVAL_INPUT = resolve_retrieval(RETRIEVAL_INPUT)
elif LEGACY_INPUT_ROOT is None:
    PREVIOUS_OUTPUT = resolve_output(PREVIOUS_OUTPUT, STAGE, required=INPUT_MODE == 'resume')
    if PREVIOUS_OUTPUT is None and INPUT_MODE == 'auto':
        from zipfile import ZipFile, BadZipFile
        diagnostics = []
        for path in INPUT.rglob('*.zip'):
            try:
                with ZipFile(path) as z:
                    if 'stage1_manifest.json' in z.namelist():
                        diagnostics.append(path)
            except BadZipFile:
                continue
        if diagnostics:
            RETRIEVAL_INPUT = resolve_retrieval()
if LEGACY_INPUT_ROOT is not None:
    LEGACY_INPUT_ROOT = Path(LEGACY_INPUT_ROOT)
    if PREVIOUS_OUTPUT is not None or INPUT_MODE == 'resume':
        raise ValueError('Không trộn legacy import với resume.')

# The old dataset mount name is irrelevant. Detect by the files actually used.
VERSION3_ROOT = find_root(VERSION3_ROOT, 'index_manifest.json',
    ['index_manifest.json', 'corpus.sqlite'], 'Version 3 index').parent
for name in ['models/models.lock.json'] + [f'models/{role}/config.json' for role in ('embedding', 'reranker', 'generator')]:
    if not (VERSION3_ROOT/name).is_file():
        raise FileNotFoundError(VERSION3_ROOT/name)
for role in ('embedding', 'reranker', 'generator'):
    if not any((VERSION3_ROOT/'models'/role).glob('*.safetensors')):
        raise FileNotFoundError(f'Thiếu trọng số {role} trong {VERSION3_ROOT}/models')
has_split = PREVIOUS_OUTPUT is not None and (PREVIOUS_OUTPUT/'data/split_manifest.json').is_file()
if DATASET_ROOT is not None or not has_split:
    DATASET_ROOT = find_root(DATASET_ROOT, 'train.json', ['train.json', 'private-official.json'], 'dataset BTC')
print('Resolved dataset:', DATASET_ROOT, '| Version 3:', VERSION3_ROOT)
print('Input mode:', INPUT_MODE, '| Retrieval-only:', RETRIEVAL_INPUT)

pins = []
for source, number in [(PREVIOUS_OUTPUT, STAGE), (UPSTREAM_OUTPUT, STAGE - 1)]:
    if source is not None:
        info = json.loads((source / f'stage{number}_manifest.json').read_text(encoding='utf-8'))
        if info.get('schema') != 2:
            raise ValueError('Input dùng schema cũ. Chọn đúng output mới hoặc legacy import ở Stage 1.')
        pins.append(info['code_commit'])
if len(set(pins)) > 1:
    raise ValueError('Upstream và previous output khác code commit.')
PIN = pins[0] if pins else (REPO_REVISION or 'main')
if pins and REPO_REVISION and REPO_REVISION != PIN:
    raise ValueError('Không đổi commit khi resume. Bắt đầu một experiment mới nếu cần đổi code.')

def available_ram_mb(proc=Path('/proc/meminfo'), cgroup=Path('/sys/fs/cgroup')):
    values = {}
    if proc.is_file():
        values = {line.split(':')[0]: int(line.split()[1])*1024
                  for line in proc.read_text().splitlines() if ':' in line}
    available = [values['MemAvailable']] if 'MemAvailable' in values else []
    for limit_file, usage_file in (('memory.max', 'memory.current'),
                                  ('memory/memory.limit_in_bytes', 'memory/memory.usage_in_bytes')):
        try:
            limit = (cgroup/limit_file).read_text().strip()
            if limit != 'max':
                available.append(max(0, int(limit)-int((cgroup/usage_file).read_text())))
        except FileNotFoundError:
            pass
    return min(available)/1024**2 if available else None


class BudgetPause(Exception):
    pass

def bounded_process(command, *, seconds=None, cwd=None, env=None):
    remaining = WORK_END - time.monotonic()
    if remaining <= 0:
        raise BudgetPause('Đã hết ngân sách phiên.')
    limit = remaining if seconds is None else min(remaining, seconds)
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(list(map(str, command)), cwd=cwd, env=env, start_new_session=True)
    stop_at = time.monotonic() + limit
    try:
        while True:
            free = available_ram_mb()
            if free is not None and free < MIN_FREE_RAM_MB / 2:
                raise MemoryError(f'RAM guard: còn {free:.0f} MiB; dừng worker để giữ snapshot.')
            left = stop_at - time.monotonic()
            if left <= 0:
                raise subprocess.TimeoutExpired(command, limit)
            try:
                rc = process.wait(timeout=min(left, 2))
                break
            except subprocess.TimeoutExpired:
                continue
    except (subprocess.TimeoutExpired, KeyboardInterrupt, MemoryError) as error:
        # Worker and every legalqa subprocess share this process group.
        # Stop all of them before hashing/exporting artifacts.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            pass
        # The group leader may exit while a GPU child ignores SIGTERM.
        # Always kill remaining group members before exporting.
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=30)
        if isinstance(error, KeyboardInterrupt):
            raise
        raise BudgetPause(str(error) if isinstance(error, MemoryError) else 'Đã dừng worker theo ngân sách; tiến độ đã ghi sẽ được export.') from error
    if rc:
        raise subprocess.CalledProcessError(rc, command)

if CODE.exists():
    if not (CODE / '.git').is_dir():
        raise RuntimeError(f'{CODE} không phải repo. Dùng phiên Kaggle mới.')
    remote = subprocess.check_output(['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True, timeout=30).strip()
    if remote.rstrip('/') != REPO_URL.rstrip('/'):
        raise RuntimeError('Repo origin không khớp.')
    dirty = subprocess.check_output(['git', '-C', str(CODE), 'status', '--porcelain'], text=True, timeout=30).strip()
    if dirty:
        raise RuntimeError('Code trong session có sửa đổi; không tự ghi đè. Dùng phiên mới.')
else:
    bounded_process(['git', 'clone', '--no-checkout', '--depth', '1', REPO_URL, CODE], seconds=300)
bounded_process(['git', '-C', CODE, 'fetch', '--depth', '1', 'origin', PIN], seconds=300)
bounded_process(['git', '-C', CODE, 'checkout', '--detach', 'FETCH_HEAD'], seconds=60)
commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True, timeout=30).strip()
if pins and commit != PIN:
    raise RuntimeError('Checkout không đúng commit đã khóa.')
if not (CODE / 'legalqa' / 'stages.py').is_file():
    raise RuntimeError('Commit chưa có stages.py. Push các thay đổi mới trước khi chạy.')
print('Pinned commit:', commit)
print('Previous:', PREVIOUS_OUTPUT, '| Upstream:', UPSTREAM_OUTPUT)

# Do not silently run an old one-GPU training config when continuing an output.
runtime_config = json.loads((CODE/'config.json').read_text(encoding='utf-8'))
if runtime_config['training'].get('world_size', 1) != 2:
    raise RuntimeError('Checkpoint khóa vào code train 1 GPU. Không đổi sang DDP giữa lượt; dùng INPUT_MODE="retrieval" cho lượt mới 2 GPU.')
if PREVIOUS_OUTPUT is not None:
    print('Exact resume: giữ commit trong manifest; các sửa mới chỉ áp dụng nếu đã có trong commit đó.')


Không dùng để resume: /kaggle/input/datasets/vphmhunhlong/legalqa-main-01-first-run/legalqa_main_stage1_v8/legalqa_main_stage1_v8_diagnostics: thiếu file hoặc sai kích thước: sft/checkpoint-120/README.md
Ưu tiên output notebook đã chọn lighth/legalqa-main-01-qlora-train: /kaggle/input/notebooks/lighth/legalqa-main-01-qlora-train/legalqa_main_stage1_v8
Resolved dataset: None | Version 3: /kaggle/input/datasets/lighth/ver3-smoke-output/legalqa_smoke_full_v1
Input mode: auto | Retrieval-only: None
Running: git clone --no-checkout --depth 1 https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git /kaggle/working/legalqa_stage_code


Cloning into '/kaggle/working/legalqa_stage_code'...


Running: git -C /kaggle/working/legalqa_stage_code fetch --depth 1 origin 6e1eb1958ba1c7ee3235860748b9eb1d177f3a57
Running: git -C /kaggle/working/legalqa_stage_code checkout --detach FETCH_HEAD


From https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa
 * branch            6e1eb1958ba1c7ee3235860748b9eb1d177f3a57 -> FETCH_HEAD


Pinned commit: 6e1eb1958ba1c7ee3235860748b9eb1d177f3a57
Previous: /kaggle/input/notebooks/lighth/legalqa-main-01-qlora-train/legalqa_main_stage1_v8 | Upstream: None
Exact resume: giữ commit trong manifest; các sửa mới chỉ áp dụng nếu đã có trong commit đó.


HEAD is now at 6e1eb19 feat: reuse verified 5600 retrieval cache and skip BM25


## Cài môi trường và kiểm tra


In [3]:
bounded_process([sys.executable, '-m', 'pip', 'install', '-q', '-r', CODE / 'requirements.txt'], seconds=1200)

# Compatibility hook lives outside CODE so exact checkpoint/source hashes remain valid.
# Source of these embedded scripts: scripts/bnb_resume_compat.py and check_bnb_resume.py.
BNB_RESUME_PATCH = r'''"""bitsandbytes 0.45.5: clear stale UVM storage metadata after checkpoint load.

The notebook embeds this file as sitecustomize.py outside the pinned legalqa
package. Thus old training/retrieval fingerprints stay valid. Only torchrun
workers install the hook. No optimizer values or hyperparameters are reset.
"""
import hashlib
import json
import os
from pathlib import Path


PATCH_ID = 'bnb-0.45.5-restored-paged-state-v1'


def restore_cuda_storage(optimizer, is_tensor):
    """Materialize restored paged tensors on the owning parameter's GPU.

    torch.save/load preserves Python is_paged/page_deviceid attributes, not
    cudaMallocManaged allocations. Ordinary CUDA tensors must not be prefetched
    as UVM. Fresh states still use the original paged optimizer allocator.
    """
    count, size, devices = 0, 0, set()
    if not optimizer.is_paged:
        return {'tensors':count, 'bytes':size, 'devices':[]}
    for group in optimizer.param_groups:
        for parameter in group['params']:
            for name, value in optimizer.state.get(parameter, {}).items():
                if not is_tensor(value) or not getattr(value, 'is_paged', False):
                    continue
                if parameter.device.type != 'cuda':
                    raise ValueError('Paged optimizer resume requires CUDA parameters')
                # detach + copy produces ordinary storage and drops stale tensor
                # attributes; dtype, shape and quantized moment bytes are retained.
                restored = value.detach().to(device=parameter.device, copy=True)
                restored.is_paged = False
                optimizer.state[parameter][name] = restored
                count += 1
                size += restored.numel()*restored.element_size()
                devices.add(str(restored.device))
    return {'tensors':count, 'bytes':size, 'devices':sorted(devices)}


def patch_optimizer_class(cls, is_tensor, report):
    original = cls.load_state_dict
    if getattr(original, '_legalqa_resume_fix', None) == PATCH_ID:
        return

    def load_state_dict(self, *args, **kwargs):
        result = original(self, *args, **kwargs)
        note = restore_cuda_storage(self, is_tensor)
        report(note)
        return result

    load_state_dict._legalqa_resume_fix = PATCH_ID
    cls.load_state_dict = load_state_dict


def install():
    from importlib.metadata import version
    installed = version('bitsandbytes')
    if installed != '0.45.5':
        raise RuntimeError(f'{PATCH_ID} requires bitsandbytes==0.45.5, found {installed}')
    import torch
    from bitsandbytes.optim.optimizer import Optimizer8bit

    def report(note):
        note = {'patch':PATCH_ID, 'patch_sha256':hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
                'bitsandbytes':installed, 'torch':torch.__version__,
                'cuda':torch.version.cuda, 'rank':int(os.environ['RANK']), **note}
        print('BNB resume storage:', json.dumps(note, sort_keys=True), flush=True)
        folder = os.environ.get('LEGALQA_BNB_RESUME_REPORT_DIR')
        if folder:
            destination = Path(folder)/f"bnb_resume_rank{note['rank']}.json"
            destination.parent.mkdir(parents=True, exist_ok=True)
            pending = destination.with_suffix('.json.tmp')
            pending.write_text(json.dumps(note, indent=2)+'\n', encoding='utf-8')
            os.replace(pending, destination)

    patch_optimizer_class(Optimizer8bit, torch.is_tensor, report)
    print(f'BNB resume compatibility installed: {PATCH_ID}; local_rank={os.environ["LOCAL_RANK"]}', flush=True)


if __name__ == 'sitecustomize' and 'LOCAL_RANK' in os.environ:
    # Python otherwise logs and ignores exceptions from sitecustomize. A missing
    # hook must stop the worker before it can hit the native CUDA abort again.
    try:
        install()
    except Exception as error:
        raise SystemExit(f'Cannot install BNB resume compatibility: {error}') from error
'''
BNB_RESUME_SMOKE = r'''"""Tiny two-GPU optimizer save/load check, before loading the LegalQA model."""
import io
import json
import os


def main():
    import torch
    import bitsandbytes as bnb
    from bitsandbytes.optim.optimizer import Optimizer8bit

    world = int(os.environ.get('WORLD_SIZE', '1'))
    rank = int(os.environ.get('LOCAL_RANK', '0'))
    if world != 2 or torch.cuda.device_count() != 2:
        raise RuntimeError('BNB resume smoke requires two visible GPU workers')
    if not getattr(Optimizer8bit.load_state_dict, '_legalqa_resume_fix', None):
        raise RuntimeError('BNB resume compatibility hook did not load in this worker')
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    # >100k elements forces bitsandbytes to allocate paged optimizer moments.
    parameter = torch.nn.Parameter(torch.full((131072,), .125, device=device))
    optimizer = bnb.optim.PagedAdamW8bit([parameter], lr=5e-5)
    parameter.grad = torch.full_like(parameter, .01)
    optimizer.step()
    if not getattr(optimizer.state[parameter]['state1'], 'is_paged', False):
        raise RuntimeError('Smoke did not exercise paged optimizer state')

    checkpoint = io.BytesIO()
    torch.save(optimizer.state_dict(), checkpoint)
    saved_parameter = parameter.detach().clone()
    parameter.grad = torch.full_like(parameter, -.02)
    optimizer.step()

    # Trainer loads directly onto args.device. This is the failure-triggering
    # path: deserialized CUDA tensors can retain stale is_paged=True attributes.
    checkpoint.seek(0)
    state = torch.load(checkpoint, map_location=device, weights_only=True)
    restored_parameter = torch.nn.Parameter(saved_parameter)
    restored_optimizer = bnb.optim.PagedAdamW8bit([restored_parameter], lr=5e-5)
    restored_optimizer.load_state_dict(state)
    for value in restored_optimizer.state[restored_parameter].values():
        if torch.is_tensor(value):
            if value.device != device or getattr(value, 'is_paged', False):
                raise RuntimeError('Restored state is not ordinary storage on the correct GPU')
    restored_parameter.grad = torch.full_like(restored_parameter, -.02)
    restored_optimizer.step()
    torch.cuda.synchronize(rank)
    torch.testing.assert_close(restored_parameter, parameter, rtol=0, atol=0)
    for name, expected in optimizer.state[parameter].items():
        actual = restored_optimizer.state[restored_parameter][name]
        if torch.is_tensor(expected):
            torch.testing.assert_close(actual.cpu(), expected.cpu(), rtol=0, atol=0)
        elif actual != expected:
            raise AssertionError(f'Restored optimizer state differs: {name}')
    print('BNB_RESUME_SMOKE_OK', json.dumps({'rank':rank, 'device':str(device),
        'step':restored_optimizer.state[restored_parameter]['step'],
        'parameters_and_optimizer_match':True}), flush=True)


if __name__ == '__main__':
    main()
'''
RUNTIME_COMPAT = WORK / 'legalqa_bnb_resume_runtime'
RUNTIME_COMPAT.mkdir(exist_ok=True)
(RUNTIME_COMPAT/'sitecustomize.py').write_text(BNB_RESUME_PATCH, encoding='utf-8')
(RUNTIME_COMPAT/'check_bnb_resume.py').write_text(BNB_RESUME_SMOKE, encoding='utf-8')
runtime_pythonpath = os.pathsep.join([str(RUNTIME_COMPAT)] + [
    value for value in os.environ.get('PYTHONPATH', '').split(os.pathsep)
    if value and value != str(RUNTIME_COMPAT)])
RUNTIME_ENV = {'PYTHONPATH':runtime_pythonpath}
print('BNB resume fix: restore saved moments on each rank GPU; keep step/optimizer values.')
bounded_process([sys.executable, '-m', 'torch.distributed.run', '--standalone',
    '--nnodes=1', '--nproc_per_node=2', RUNTIME_COMPAT/'check_bnb_resume.py'],
    cwd=CODE, seconds=180, env={**os.environ, **RUNTIME_ENV,
        'LEGALQA_BNB_RESUME_REPORT_DIR':str(WORK/'bnb_resume_smoke')})

if STAGE == 2:
    bounded_process([sys.executable, '-m', 'nltk.downloader', '-q', 'wordnet', 'omw-1.4'], seconds=300)
    bounded_process([sys.executable, 'scripts/check_metrics.py'], cwd=CODE, seconds=300)
bounded_process([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CODE, seconds=300)


Running: /usr/bin/python3 -m pip install -q -r /kaggle/working/legalqa_stage_code/requirements.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.1/163.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.30.2 which is incompatible.


BNB resume fix: restore saved moments on each rank GPU; keep step/optimizer values.
Running: /usr/bin/python3 -m torch.distributed.run --standalone --nnodes=1 --nproc_per_node=2 /kaggle/working/legalqa_bnb_resume_runtime/check_bnb_resume.py



*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W916 10:56:22.997032774 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


BNB resume compatibility installed: bnb-0.45.5-restored-paged-state-v1; local_rank=1
BNB resume compatibility installed: bnb-0.45.5-restored-paged-state-v1; local_rank=0
BNB resume storage: {"bitsandbytes": "0.45.5", "bytes": 262144, "cuda": "12.8", "devices": ["cuda:1"], "patch": "bnb-0.45.5-restored-paged-state-v1", "patch_sha256": "1610c9a74ebb1eba1f676f8380ab605e1c8dc8d56bfb38402691dbd7ebc638d8", "rank": 1, "tensors": 2, "torch": "2.10.0+cu128"}
BNB resume storage: {"bitsandbytes": "0.45.5", "bytes": 262144, "cuda": "12.8", "devices": ["cuda:0"], "patch": "bnb-0.45.5-restored-paged-state-v1", "patch_sha256": "1610c9a74ebb1eba1f676f8380ab605e1c8dc8d56bfb38402691dbd7ebc638d8", "rank": 0, "tensors": 2, "torch": "2.10.0+cu128"}
BNB_RESUME_SMOKE_OK {"rank": 1, "device": "cuda:1", "step": 2, "parameters_and_optimizer_match": true}
BNB_RESUME_SMOKE_OK {"rank": 0, "device": "cuda:0", "step": 2, "parameters_and_optimizer_match": true}
Running: /usr/bin/python3 -B -m unittest discover -s tes

...

SNAPSHOT paused: /tmp/tmpw0yuwa1c/source/stage3_manifest.json
Diagnostics: /tmp/tmpw0yuwa1c/source/legalqa_main_stage3_v8_diagnostics.zip
Progress: {'complete': False, 'phase': 'generate', 'answered': 1, 'total': 2}
Progress: {'complete': True, 'phase': 'submission_ready', 'answers': 2, 'selected_meteor': 0.5}
Evaluation objective: {'primary_metric': 'meteor', 'secondary_metric': 'rougeL', 'target_meteor': 0.65}
Retrieval: {'bm25_cache_mb': 1024, 'fast_phrase_precise': True, 'phrase_cache_mb': 64, 'phrase_workers': 2, 'phrase_reuse_readers': True, 'phrase_mmap_mb': 1024, 'bm25_k': 100, 'dense_k': 100, 'rrf_constant': 60, 'precise_bm25_k': 40, 'pool_k': 32, 'max_children_per_parent': 2, 'parents_k': 4, 'lexical_score_weight': 2.0, 'phrase_match_bonus': 1.0, 'exact_document_bonus': 4.0, 'year_match_bonus': 1.0, 'recency_bonus': 0.75, 'embedding_batch': 32, 'reranker_batch': 8, 'reranker_max_tokens': 768} Generation: {'max_input_tokens': 4096, 'max_new_tokens': 1536, 'parent_max_tokens': 

ok
test_stage_launches_torchrun_only_for_fit (test_stages.StageTests.test_stage_launches_torchrun_only_for_fit) ... ok
test_three_notebooks_share_supervisor_setup_and_runner (test_stages.StageTests.test_three_notebooks_share_supervisor_setup_and_runner) ... ok
test_timeout_cannot_publish_complete_from_old_progress (test_stages.StageTests.test_timeout_cannot_publish_complete_from_old_progress) ... ok
test_training_callback_saves_once_and_keeps_completed_epoch (test_stages.StageTests.test_training_callback_saves_once_and_keeps_completed_epoch) ... ok
test_training_reuse_rejects_changed_config_and_split (test_stages.StageTests.test_training_reuse_rejects_changed_config_and_split) ... ok
test_training_uses_two_workers_with_unchanged_effective_batch (test_stages.StageTests.test_training_uses_two_workers_with_unchanged_effective_batch) ... ok
test_fp16_frozen_head_loss_gradients_and_global_token_denominator (test_training_memory.FusedLossCudaTests.test_fp16_frozen_head_loss_gradients_and_glo

QLoRA loss=liger_fused_linear_cross_entropy; full targets; no CPU offload
QLoRA loss=liger_fused_linear_cross_entropy; full targets; no CPU offload


ok
test_compaction_preserves_targets_masks_order_and_padding (test_training_memory.TrainingMemoryTests.test_compaction_preserves_targets_masks_order_and_padding) ... ok
test_patch_is_instance_local_and_preserves_loss_kwargs (test_training_memory.TrainingMemoryTests.test_patch_is_instance_local_and_preserves_loss_kwargs) ... ok
test_rejects_unsupported_head_before_training (test_training_memory.TrainingMemoryTests.test_rejects_unsupported_head_before_training) ... ok

----------------------------------------------------------------------
Ran 104 tests in 32.647s

OK


QLoRA loss=liger_fused_linear_cross_entropy; full targets; no CPU offload


## Chạy trong ngân sách và export cả tiến độ dở dang

Mọi xử lý/kiểm tra artifact dùng legalqa/stages.py chung cho ba notebook. Diagnostics chứa dữ liệu dev, reference, retrieval, prediction, audit, metrics và trạng thái train đã có; không chứa trọng số.


In [4]:
if 'private-official.json' not in (CODE / 'legalqa/stages.py').read_text(encoding='utf-8'):
    raise RuntimeError('Pinned code still uses public data. Push the private update and start a new run; do not resume old public outputs.')

RUN_ROOT = WORK / f'legalqa_main_stage{STAGE}_v8'
OPTIONS = WORK / f'legalqa_stage{STAGE}_options.json'
options = {
    'stage': STAGE, 'root': str(RUN_ROOT), 'version3': str(VERSION3_ROOT),
    'dataset': str(DATASET_ROOT) if DATASET_ROOT is not None else None, 'mode': MODE, 'max_new_questions': MAX_NEW_QUESTIONS,
    'previous': str(PREVIOUS_OUTPUT) if PREVIOUS_OUTPUT is not None else None,
    'upstream': str(UPSTREAM_OUTPUT) if UPSTREAM_OUTPUT is not None else None,
    'legacy': str(LEGACY_INPUT_ROOT) if LEGACY_INPUT_ROOT is not None else None,
    'retrieval_input': str(RETRIEVAL_INPUT) if RETRIEVAL_INPUT is not None else None,
}
OPTIONS.write_text(json.dumps(options, ensure_ascii=False, indent=2), encoding='utf-8')
# Cooperative pause 10 minutes before hard worker stop, for saving Trainer state.
remaining = max(0, WORK_END - time.monotonic())
worker_env = {**os.environ, **RUNTIME_ENV,
              'LEGALQA_BNB_RESUME_REPORT_DIR': str(RUN_ROOT/'runtime_compat'), 'LEGALQA_DEADLINE': str(time.time() + max(0, remaining - 600)),
              'PYTHONUNBUFFERED': '1', 'LEGALQA_MIN_FREE_RAM_MB': str(MIN_FREE_RAM_MB),
              'TOKENIZERS_PARALLELISM': 'false'}
outcome, failure = 'ok', None
try:
    bounded_process([sys.executable, '-m', 'legalqa.stages', 'run', '--options', OPTIONS],
                    cwd=CODE, env=worker_env)
except BudgetPause as error:
    outcome = 'paused'
    print(str(error), flush=True)
except Exception as error:
    outcome, failure = 'failed', error
finally:
    if (RUN_ROOT / 'session.json').is_file():
        # Export runs only after the entire compute process group has stopped.
        # It is bounded separately, without restarting the 9h compute budget.
        original_end = WORK_END
        WORK_END = min(SESSION_STARTED + 10 * 3600, time.monotonic() + EXPORT_SECONDS)
        try:
            bounded_process([sys.executable, '-m', 'legalqa.stages', 'finalize',
                             '--options', OPTIONS, '--outcome', outcome], cwd=CODE)
        finally:
            WORK_END = original_end
if failure is not None:
    raise failure
manifest_path = RUN_ROOT / f'stage{STAGE}_manifest.json'
if not manifest_path.is_file():
    raise RuntimeError('Chưa tạo được snapshot; xem lỗi setup ở trên.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('STATUS:', manifest['status'])
print('PROGRESS:', manifest['progress'])
print('OUTPUT:', RUN_ROOT)
if manifest['status'] == 'complete':
    print('Stage hoàn tất. Có thể dùng output cho stage tiếp theo.')
else:
    print('Phiên kết thúc có chủ đích. Save output, Add Input vào CÙNG notebook, rồi chạy tiếp.')
proof_path = RUN_ROOT / 'sft/distributed_training.json'
if proof_path.is_file():
    print('DDP proof:', json.loads(proof_path.read_text(encoding='utf-8')))
print('Lần sau: giữ Input Version 3 (index/models), Add Input toàn bộ OUTPUT ở trên; INPUT_MODE=auto.')
print('Diagnostics ZIP không có trọng số/optimizer, không thay thế output đầy đủ để resume.')


Running: /usr/bin/python3 -m legalqa.stages run --options /kaggle/working/legalqa_stage1_options.json
Evaluation objective: {'primary_metric': 'meteor', 'secondary_metric': 'rougeL', 'target_meteor': 0.65}
Retrieval: {'bm25_cache_mb': 1024, 'fast_phrase_precise': True, 'phrase_cache_mb': 64, 'phrase_workers': 2, 'phrase_reuse_readers': True, 'phrase_mmap_mb': 1024, 'bm25_k': 100, 'dense_k': 100, 'rrf_constant': 60, 'precise_bm25_k': 40, 'pool_k': 32, 'max_children_per_parent': 2, 'parents_k': 4, 'lexical_score_weight': 2.0, 'phrase_match_bonus': 1.0, 'exact_document_bonus': 4.0, 'year_match_bonus': 1.0, 'recency_bonus': 0.75, 'embedding_batch': 32, 'reranker_batch': 8, 'reranker_max_tokens': 768} Generation: {'max_input_tokens': 4096, 'max_new_tokens': 1536, 'parent_max_tokens': 1400, 'min_context_tokens': 256, 'load_in_4bit': True}
Running: /usr/bin/python3 -m torch.distributed.run --standalone --nnodes=1 --nproc_per_node=2 --module legalqa --config /kaggle/working/legalqa_main_stage1


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W916 10:57:19.466272801 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


BNB resume compatibility installed: bnb-0.45.5-restored-paged-state-v1; local_rank=1
BNB resume compatibility installed: bnb-0.45.5-restored-paged-state-v1; local_rank=0


[W916 10:57:39.541262820 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 10:57:39.541507368 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W916 10:57:44.046693709 ProcessGroupNCCL.cpp:5138] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()


QLoRA rank=0/2 local_rank=0 device=cuda:0 GPU=Tesla T4 accumulation=4 effective_batch=8QLoRA rank=1/2 local_rank=1 device=cuda:1 GPU=Tesla T4 accumulation=4 effective_batch=8



...

{'loss': 0.5557, 'grad_norm': 0.5288335084915161, 'learning_rate': 6.974390731606661e-11, 'epoch': 2.0}


100%|██████████| 1400/1400 [1:21:04<00:00,  3.47s/it]


{'train_runtime': 4864.2499, 'train_samples_per_second': 2.303, 'train_steps_per_second': 0.288, 'train_loss': 0.07520639836788177, 'epoch': 2.0}


/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


{
  "trained_examples": 5600,
  "skipped": 0,
  "final_adapter": "/kaggle/working/legalqa_main_stage1_v8/sft/adapter_last"
}
Progress: {'complete': True, 'phase': 'training_complete'}
Running: /usr/bin/python3 -m legalqa.stages finalize --options /kaggle/working/legalqa_stage1_options.json --outcome ok
SNAPSHOT complete: /kaggle/working/legalqa_main_stage1_v8/stage1_manifest.json
Diagnostics: /kaggle/working/legalqa_main_stage1_v8/legalqa_main_stage1_v8_diagnostics.zip
STATUS: complete
PROGRESS: {'complete': True, 'phase': 'training_complete'}
OUTPUT: /kaggle/working/legalqa_main_stage1_v8
Stage hoàn tất. Có thể dùng output cho stage tiếp theo.
DDP proof: {'world_size': 2, 'workers': [{'rank': 0, 'device': 'cuda:0', 'gpu': 'Tesla T4', 'global_step': 1400, 'peak_allocated_bytes': 6366918656}, {'rank': 1, 'device': 'cuda:1', 'gpu': 'Tesla T4', 'global_step': 1400, 'peak_allocated_bytes': 5358295552}], 'effective_batch_size': 8}
Lần sau: giữ Input Version 3 (index/models), Add Input toàn 